# Step 1 — Teacher Labeling with Qwen2.5-32B (CoT reasoning chains over GECS taxonomy)

**Runtime:** Runtime → Change runtime type → **A100 GPU**.

**Inputs you'll upload:**
- `task1_train.csv`
- `gecs_taxonomy.json`  (the 145 official Morningstar GECS definitions)

**Outputs downloaded directly from Colab at the end:**
- `reasoning_chains.jsonl` — one row per labeled training example
- `teacher_progress.json` — index of how far we got (resume-capable)

**Budget:** ~12 A100-hours for 3000 labeled examples. Resume-capable inside the current Colab session. Download the output zip before closing the runtime.

In [ ]:
# ── 1. Setup ─────────────────────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

!pip install -q vllm transformers accelerate bitsandbytes

import os
OUT_DIR = '/content/gecs_distill'
os.makedirs(OUT_DIR, exist_ok=True)
print(f'Output dir: {OUT_DIR}')
print('This notebook uses direct upload + direct download. No Drive mount required.')

In [ ]:
# ── 2. Upload input CSV + taxonomy ──────────────────────────────────────
from google.colab import files
print('Upload your Task 1 CSV and gecs_taxonomy.json (multi-select):')
uploaded = files.upload()

import pandas as pd, json
csv_files = [name for name in uploaded if name.lower().endswith('.csv')]
json_files = [name for name in uploaded if name.lower().endswith('.json')]
if not csv_files:
    raise RuntimeError('Upload one Task 1 CSV file.')
if 'gecs_taxonomy.json' in uploaded:
    taxonomy_file = 'gecs_taxonomy.json'
elif json_files:
    taxonomy_file = json_files[0]
else:
    raise RuntimeError('Upload gecs_taxonomy.json.')
input_csv = csv_files[0]
train = pd.read_csv(input_csv)
tax = json.load(open(taxonomy_file, encoding='utf-8'))

def build_teacher_text(df):
    if 'text' in df.columns:
        return df['text'].fillna('').astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()
    required = ['SegmentName', 'SegmentDescription']
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise RuntimeError(f'Missing text columns: {missing}. Need either text or SegmentName + SegmentDescription.')
    return (
        df['SegmentName'].fillna('').astype(str).str.strip() + ' ' +
        df['SegmentDescription'].fillna('').astype(str).str.strip()
    ).str.replace(r'\s+', ' ', regex=True).str.strip()

train['teacher_text'] = build_teacher_text(train)
train = train[train['teacher_text'].str.len() > 0].copy()

# Build code -> definition lookup
tax_by_code = {e['mstar_code']: e for e in tax}
print(f'input file: {input_csv}')
print(f'train rows: {len(train):,}  taxonomy entries: {len(tax)}')

In [ ]:
# ── 3. Sample 3000 training rows (stratified by class) ──────────────────
import numpy as np

N_PER_CLASS = 21   # 21 × 145 ≈ 3045 samples

def norm_code(v): return str(int(v)).zfill(8)
train['code'] = train['mstar_code'].map(norm_code)

pieces = []
for code, grp in train.groupby('code'):
    n = min(N_PER_CLASS, len(grp))
    pieces.append(grp.sample(n, random_state=42))
subset = pd.concat(pieces, ignore_index=True)
subset = subset.sample(frac=1, random_state=7).reset_index(drop=True)
print(f'Sampled subset: {len(subset):,} rows across {subset["code"].nunique()} classes')

In [ ]:
# ── 4. Build CoT prompt with top-K candidate GECS definitions ───────────
# Heuristic shortlist: for each example, the prompt lists the 5 most
# plausible candidates (its true label + 4 random sector-mates).
# The teacher must reason through them and pick. This forces grounding.
import random
random.seed(42)

by_sector = {}
for c, e in tax_by_code.items():
    by_sector.setdefault(c[:3], []).append(c)

PROMPT = '''You are a senior Morningstar GECS analyst classifying companies into the 145-code industry taxonomy.

COMPANY SEGMENT TEXT:
{text}

CANDIDATE GECS INDUSTRY CODES (5 plausible options — pick the best match):
{candidates}

Reason step by step.
1. Identify the dominant economic activity from the segment text.
2. Map it to a Morningstar Super Sector (Cyclical / Defensive / Sensitive).
3. Narrow to one of the 11 sectors based on the activity keywords.
4. Pick the industry group, then the specific 8-digit code.
5. Justify by quoting the matching phrase from the official GECS definition.

End your response with exactly this line (and nothing after it):
FINAL_CODE: <8-digit code>
'''

def build_candidates(true_code: str) -> str:
    sect = true_code[:3]
    sect_pool = [c for c in by_sector.get(sect, []) if c != true_code]
    random.shuffle(sect_pool)
    chosen = [true_code] + sect_pool[:4]
    random.shuffle(chosen)
    lines = []
    for c in chosen:
        e = tax_by_code[c]
        lines.append(f"  - {c} [{e['sector_name']}] {e['industry_name']}: {e['description'][:280]}")
    return '\n'.join(lines)

# Inspect one prompt
row = subset.iloc[0]
print(PROMPT.format(text=row['teacher_text'][:1500], candidates=build_candidates(row['code']))[:2500])

In [ ]:
# ── 5. Load Qwen2.5-32B-Instruct at 4-bit via vLLM ──────────────────────
# If A100-40GB is too tight, fall back to Qwen2.5-14B-Instruct (no quant).
from vllm import LLM, SamplingParams

MODEL = 'Qwen/Qwen2.5-32B-Instruct'      # change to '14B-Instruct' if OOM
QUANT = 'awq_marlin'                      # AWQ + Marlin kernel for A100

# If 32B AWQ isn't available locally on Colab, comment out and use bnb:
# MODEL = 'Qwen/Qwen2.5-14B-Instruct'
# QUANT = None

llm = LLM(
    model=MODEL,
    quantization='awq' if QUANT else None,
    dtype='float16',
    gpu_memory_utilization=0.90,
    max_model_len=4096,
    trust_remote_code=True,
)

sampling = SamplingParams(
    temperature=0.2,
    top_p=0.9,
    max_tokens=600,
    stop=['\n\n\n'],
)
print('Teacher loaded.')

In [ ]:
# ── 6. Resume-capable batched generation ────────────────────────────────
import json, time, re
from tqdm.auto import tqdm

OUT_FILE = f'{OUT_DIR}/reasoning_chains.jsonl'
PROG_FILE = f'{OUT_DIR}/teacher_progress.json'

done_idx = set()
if os.path.exists(OUT_FILE):
    with open(OUT_FILE) as f:
        for line in f:
            try: done_idx.add(json.loads(line)['idx'])
            except: pass
print(f'Already done: {len(done_idx):,} / {len(subset):,}')

BATCH = 16
remaining = [i for i in range(len(subset)) if i not in done_idx]
print(f'Remaining: {len(remaining):,}')

code_re = re.compile(r'FINAL_CODE:\s*(\d{8})')

with open(OUT_FILE, 'a') as f:
    for b_start in tqdm(range(0, len(remaining), BATCH), desc='teacher'):
        idxs = remaining[b_start:b_start + BATCH]
        prompts = []
        for i in idxs:
            row = subset.iloc[i]
            prompts.append(PROMPT.format(
                text=str(row['teacher_text'])[:1800],
                candidates=build_candidates(row['code'])
            ))
        outs = llm.generate(prompts, sampling, use_tqdm=False)
        for i, out in zip(idxs, outs):
            row = subset.iloc[i]
            text = out.outputs[0].text
            m = code_re.search(text)
            pred = m.group(1) if m else None
            f.write(json.dumps({
                'idx': int(i),
                'text': str(row['teacher_text']),
                'true_code': row['code'],
                'reasoning': text.strip(),
                'teacher_pred': pred,
                'teacher_correct': pred == row['code'],
            }) + '\n')
        f.flush()

with open(PROG_FILE, 'w') as f:
    json.dump({'completed': len(subset), 'total': len(subset)}, f)
print('Done.')
zip_path = '/content/gecs_distill_outputs.zip'
!zip -qr "$zip_path" "$OUT_DIR"
files.download(zip_path)

In [ ]:
# ── 7. Quality check on the teacher's labels ───────────────────────────
rows = [json.loads(l) for l in open(OUT_FILE)]
n_match = sum(1 for r in rows if r['teacher_correct'])
print(f'Teacher → ground-truth agreement: {n_match}/{len(rows)} = {100*n_match/len(rows):.2f}%')
print(f'(>=80% means the teacher is producing high-quality CoT chains.)')

# Show three examples
for r in rows[:3]:
    print('=' * 80)
    print(f'TRUE: {r["true_code"]}   TEACHER PRED: {r["teacher_pred"]}')
    print(r['reasoning'][:1500])
    print()